# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. All dataset entities are referenced by their `@id` fields, following the Croissant standard.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's list the available record sets, fields, and their `@id`s in the dataset. This ensures we can reference them accurately when accessing or manipulating data.

In [ ]:
# List all available record sets and their fields by `@id`
from pprint import pprint

rs_list = []
for record_set in metadata.record_sets:
    print(f"Record Set: {record_set.name} (id: {record_set.id})")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (id: {field.id}, dtype: {field.data_type})")
    rs_list.append(record_set.id)
print("\nAll Record Set @ids:")
pprint(rs_list)


## 3. Data Extraction
Let's extract data from one or more record sets into DataFrames for analysis. We will use record set and field `@id`s gathered above.

In [ ]:
# If your dataset has at least one record set, list their @ids in record_sets
record_sets = rs_list  # From previous overview
dataframes = {}

# Load the records from each record set
for record_set_id in record_sets:
    print(f"\nLoading records for Record Set @id: {record_set_id}")
    # Each record set is loaded as a generator of dicts
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    else:
        print("No records found.")


## 4. Exploratory Data Analysis (EDA)
Let's process one record set with a numeric field. We'll filter, normalize, and optionally group the data. All references use Croissant `@id`s.

In [ ]:
# Choose a record set for EDA
if dataframes:
    # Pick the first data-rich record set
    record_set_id = next(iter(dataframes.keys()))
    df = dataframes[record_set_id]
    print(f"Using Record Set @id: {record_set_id}")
    # Try to select a numeric field
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Selected numeric field (@id): {numeric_field}")
        # Filtering
        threshold = df[numeric_field].mean()  # Use mean as threshold for example
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization (z-score)
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a likely categorical field
        group_field_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            print(f"Grouping by field (@id): {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            display(grouped_df.head())
        else:
            print("No suitable categorical field to group by.")
    else:
        print("No numeric fields found in this record set.")
else:
    print("No dataframes available. Check data extraction above.")

## 5. Visualization
Visualizing data distributions or relationships between fields helps understand your dataset. Let's plot a histogram for the selected numeric field, grouped by a categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals():
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field if available
    if 'group_field' in locals():
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 mass spectrometry dataset using the `mlcroissant` library, explored its structure using `@id` fields, extracted data for analysis, and performed simple EDA and visualization. This workflow demonstrates how record sets and fields described by Croissant can be systematically explored for reproducible science.